# Tarea 1: Limpieza y Análisis Inicial de Datos con Python y Pandas
**Seminario de Sistemas 2 — 2do Semestre 2026**  
**Grupo 16**

---

### Integrantes:
* **César Rolando Hernández Palacios** — Carnet: `202000806`
* **Keneth Willard Lopez Ovalle** — Carnet: `202100106`

---

### Descripción General
El objetivo de este notebook es documentar y ejecutar el flujo integral de limpieza, normalización, tratamiento de inconsistencias y análisis exploratorio del dataset `dataset_sucio.csv`. Se abordan las dimensiones de calidad de datos establecidas en el estándar **ISO/IEC 25012** (Completitud, Unicidad, Consistencia, Exactitud sintáctica y Temporalidad) y se presentan tablas pivote comparativas del estado *Antes* vs. *Después*.

## 1. Carga del Dataset y Diccionario de Datos

Iniciamos importando las librerías esenciales (`pandas` y `numpy`) y cargando el archivo `dataset_sucio.csv`. Asimismo, preservamos una copia inmutable (`df_original`) para realizar comparaciones de calidad y tablas pivote del estado *Antes* vs. *Después*.

### Diccionario de Datos:
| Columna | Tipo Esperado | Descripción |
|---|---|---|
| `id_cliente` | Entero (`int64`) | Identificador numérico único del cliente |
| `nombre` | Texto (`object`) | Nombre completo del cliente |
| `genero` | Categórico (`object` $\to$ `F`/`M`) | Género del cliente |
| `fecha_registro` | Temporal (`datetime64`) | Fecha en que el cliente se registró en el sistema |
| `gasto_q` | Numérico (`float64`) | Gasto total acumulado en Quetzales (Q) |
| `ciudad` | Categórico (`object`) | Ciudad guatemalteca de residencia |
| `categoria` | Categórico (`object`) | Categoría de compra o servicio |

In [1]:
import pandas as pd
import numpy as np

# Carga del dataset original
df = pd.read_csv("dataset_sucio.csv")

# Preservamos una copia intacta del estado original
df_original = df.copy()

print(f"Dimensiones iniciales: {df.shape[0]} filas x {df.shape[1]} columnas")

Dimensiones iniciales: 5000 filas x 7 columnas


## 2. Exploración del Estado Original (Antes)

Revisamos las primeras filas, la información de tipos de datos y el resumen estadístico descriptivo general para diagnosticar anomalías en el dataset sucio.

In [2]:
# Vista previa de las primeras 10 filas
df.head(10)

,id_cliente,nombre,genero,fecha_registro,gasto_q,ciudad,categoria
0,502609,Paola Morales,m,2026-02-27,NaN,Antigua,Retail
1,500588,ANA DIAZ,f,2026-02-24,371.80,Villa Nueva,Services
2,502408,Maria Paz,m,2026-01-15,"373,33",quetzaltenango,retail
3,501688,Ana Mejia,f,2026-01-18,361.23,Quetzaltenango,Education
4,502916,Carlos Mendez,f,29/01/2026,"339,01",Amatitlan,Retail
5,504200,Oscar Rojas,m,2026-01-09,138.22,Antigua,RETAIL
6,502029,Isabel Ramirez,m,2026-02-05,"166,31",Villa Nueva,SERVICES
7,501366,Pedro Herrera,m,12/02/2026,"405,62",Amatitlan,Education
8,502614,Patricia Soto,F,2026-01-11,222.88,guatemala,Retail
9,502794,Pablo Perez,M,2026-02-06,"287,37",Escuintla,Retail


In [3]:
# Información estructural del DataFrame original
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5000 entries, 0 to 4999
Data columns (total 7 columns):
 #   Column          Non-Null Count  Dtype 
---  ------          --------------  ----- 
 0   id_cliente      5000 non-null   int64 
 1   nombre          5000 non-null   object
 2   genero          4940 non-null   object
 3   fecha_registro  5000 non-null   object
 4   gasto_q         4495 non-null   object
 5   ciudad          4843 non-null   object
 6   categoria       5000 non-null   object
dtypes: int64(1), object(6)
memory usage: 273.6+ KB


In [4]:
# Resumen descriptivo general incluyendo variables cualitativas
df.describe(include='all')

,id_cliente,nombre,genero,fecha_registro,gasto_q,ciudad,categoria
count,5000.00000,5000,4940,5000,4495,4843,5000
unique,NaN,2864,7,120,4301,41,16
top,NaN,Ricardo Vargas,M,2026-01-02,479.28,Quetzaltenango,Food
freq,NaN,8,834,71,4,328,674
mean,502449.09300,NaN,NaN,NaN,NaN,NaN,NaN
std,1415.02615,NaN,NaN,NaN,NaN,NaN,NaN
min,500001.00000,NaN,NaN,NaN,NaN,NaN,NaN
25%,501220.75000,NaN,NaN,NaN,NaN,NaN,NaN
50%,502451.50000,NaN,NaN,NaN,NaN,NaN,NaN
75%,503674.25000,NaN,NaN,NaN,NaN,NaN,NaN


## 3. Tratamiento de Registros Duplicados (Unicidad)

Evaluamos la presencia de filas exactamente repetidas (`duplicated()`). La duplicidad de registros altera las métricas descriptivas y sesga los modelos analíticos.

In [5]:
# Conteo de duplicados exactos
num_dup = df.duplicated().sum()
print(f"Total de registros duplicados detectados: {num_dup}")

# Visualización de una muestra de registros duplicados antes de su remoción
df[df.duplicated(keep=False)].sort_values('id_cliente').head(10)

Total de registros duplicados detectados: 100


,id_cliente,nombre,genero,fecha_registro,gasto_q,ciudad,categoria
4536,500011,Gabriela Dominguez,F,2026-01-25,"354,39",Amatitlan,Education
4071,500011,Gabriela Dominguez,F,2026-01-25,"354,39",Amatitlan,Education
117,500028,DIANA FUENTES,F,2026-01-17,NaN,Quetzaltenango,Services
807,500028,DIANA FUENTES,F,2026-01-17,NaN,Quetzaltenango,Services
311,500056,Isabel Castillo,f,2026-02-18,"468,48",AMATITLAN,Services
1665,500056,Isabel Castillo,f,2026-02-18,"468,48",AMATITLAN,Services
889,500084,ISABEL ALVAREZ,M,2026-01-13,59.46,Villa Nueva,Food
4892,500084,ISABEL ALVAREZ,M,2026-01-13,59.46,Villa Nueva,Food
4495,500122,rene lopez,f,2026-01-02,87.19,Antigua,Education
4072,500122,rene lopez,f,2026-01-02,87.19,Antigua,Education


In [6]:
# Eliminación de duplicados manteniendo la primera ocurrencia
df = df.drop_duplicates()
print(f"Dimensiones tras drop_duplicates(): {df.shape[0]} filas x {df.shape[1]} columnas")

Dimensiones tras drop_duplicates(): 4900 filas x 7 columnas


## 4. Detección y Tratamiento de Nulos y Espacios en Blanco (Completitud)

> **Hallazgo de Calidad (Bug B - Espacios Ocultos):**  
> En el dataset existen celdas que contienen únicamente espacios en blanco (ej. `' '`, `'   '`). Por defecto, `pd.read_csv` no las interpreta como `NaN`. Si se ejecuta `dropna()` antes de limpiar los espacios, estas celdas sobreviven como cadenas vacías (`''`), generando registros inconsistentes.
>
> **Solución aplicada:** Normalizar espacios con `.str.strip()` y mapear cadenas vacías a `np.nan` antes de aplicar `dropna()`. De esta manera, se revelan los nulos reales:
> - `genero`: 113 nulos (60 NaN originales + 53 celdas de espacios en blanco)
> - `ciudad`: 222 nulos (152 NaN originales + 70 celdas de espacios en blanco)
> - `gasto_q`: 497 nulos

In [7]:
# Conteo de nulos detectados inicialmente por pandas
nulos_explicitos = df.isnull().sum()
print("--- Nulos reportados directamente por isnull() ---")
print(nulos_explicitos)

# Normalización de espacios y conversión de cadenas vacías a NaN
for col in df.select_dtypes(include='object').columns:
    df[col] = df[col].astype(str).str.strip()
    df[col] = df[col].replace({'nan': np.nan, '': np.nan})

nulos_reales = df.isnull().sum()
nulos_espacios = nulos_reales - nulos_explicitos

print("\n--- Celdas que solo contenían espacios en blanco (ocultas a isnull) ---")
print(nulos_espacios[nulos_espacios > 0])

print("\n--- Nulos reales tras normalizar espacios y mapear cadenas vacías a NaN ---")
print(nulos_reales)

--- Nulos reportados directamente por isnull() ---
id_cliente          0
nombre              0
genero             60
fecha_registro      0
gasto_q           497
ciudad            152
categoria           0
dtype: int64

--- Celdas que solo contenían espacios en blanco (ocultas a isnull) ---
genero    53
ciudad    70
dtype: int64

--- Nulos reales tras normalizar espacios y mapear cadenas vacías a NaN ---
id_cliente          0
nombre              0
genero            113
fecha_registro      0
gasto_q           497
ciudad            222
categoria           0
dtype: int64


In [8]:
# Aplicación de dropna para eliminar filas incompletas
df = df.dropna()
print(f"Dimensiones tras dropna(): {df.shape[0]} filas x {df.shape[1]} columnas")
print("\nVerificación de nulos restantes (debe ser 0 en todas las columnas):")
print(df.isnull().sum())

Dimensiones tras dropna(): 4105 filas x 7 columnas

Verificación de nulos restantes (debe ser 0 en todas las columnas):
id_cliente        0
nombre            0
genero            0
fecha_registro    0
gasto_q           0
ciudad            0
categoria         0
dtype: int64


## 5. Estandarización de Valores y Formatos (Consistencia y Exactitud)

> **Hallazgo de Calidad (Bug A - Parseo de Fechas):**  
> El dataset contiene fechas en dos formatos: ISO (`YYYY-MM-DD`) y barra inclinada (`DD/MM/YYYY`).  
> Al inspeccionar las fechas en slash, observamos que el segundo campo únicamente toma valores `01`, `02` y `03` (meses de enero a marzo de 2026), mientras que el primer campo va de `01` a `31` (días).  
> Si se utiliza `dayfirst=False`, pandas asume `MM/DD/YYYY`, transformando fechas como `12/02/2026` (12 de febrero) en `2026-12-02` (2 de diciembre), corrompiendo 412 filas hacia meses 04–12 inexistentes en el origen.
>
> **Solución:** Configurar `pd.to_datetime(..., format='mixed', dayfirst=True)` y validar con aserción que el rango mensual se encuentre estrictamente entre enero y marzo de 2026.

Transformaciones realizadas:
1. `nombre`: Formato título (`str.title()`) y colapso de espacios múltiples (`str.replace(r'\s+', ' ', regex=True)`).
2. `genero`: Mayúsculas canónicas (`'M'`, `'F'`).
3. `ciudad`: Formato título (`str.title()`) para colapsar variantes como `'AMATITLAN'`, `'amatitlan'`, `' Amatitlan '`.
4. `categoria`: Formato título (`str.title()`).
5. `gasto_q`: Sustitución de coma decimal por punto (`','` $\to$ `'.'`) y conversión a tipo `float64`.
6. `fecha_registro`: Conversión a `datetime64[ns]` con `dayfirst=True`.

In [9]:
# Estandarización de columnas de texto
df['nombre'] = df['nombre'].str.title().str.replace(r'\s+', ' ', regex=True)
df['genero'] = df['genero'].str.upper()
df['ciudad'] = df['ciudad'].str.title()
df['categoria'] = df['categoria'].str.title()

# Estandarización numérica de gasto_q
df['gasto_q'] = df['gasto_q'].str.replace(',', '.', regex=False).astype(float)

# Estandarización de fecha_registro con dayfirst=True
df['fecha_registro'] = pd.to_datetime(df['fecha_registro'], format='mixed', dayfirst=True)

# Validaciones de integridad del dataset depurado
assert df['fecha_registro'].dt.month.between(1, 3).all(),     "Error: existen meses fuera del rango esperado (enero a marzo)"
assert df.isnull().sum().sum() == 0,     "Error: quedan valores nulos tras la depuración"
assert not (df.select_dtypes(include='object') == '').any().any(),     "Error: quedan celdas vacías ocultas (cadenas de longitud cero)"
assert df.duplicated().sum() == 0,     "Error: quedan registros duplicados"
assert set(df['genero'].unique()) == {'F', 'M'},     f"Error: géneros no canónicos -> {sorted(df['genero'].unique())}"
assert df['ciudad'].nunique() == 8,     f"Error: se esperaban 8 ciudades canónicas, se obtuvieron {df['ciudad'].nunique()}"

print("Las 6 aserciones de integridad pasaron satisfactoriamente.")
print(f"Registros finales depurados: {df.shape[0]:,}")

Las 6 aserciones de integridad pasaron satisfactoriamente.


Registros finales depurados: 4,105


## 6. Exploración del Estado Depurado (Después)

Inspeccionamos nuevamente la estructura, primeras filas y estadísticas del DataFrame limpio para verificar la tipificación y completitud.

In [10]:
# Vista previa de las primeras 10 filas depuradas
df.head(10)

,id_cliente,nombre,genero,fecha_registro,gasto_q,ciudad,categoria
1,500588,Ana Diaz,F,2026-02-24,371.80,Villa Nueva,Services
2,502408,Maria Paz,M,2026-01-15,373.33,Quetzaltenango,Retail
3,501688,Ana Mejia,F,2026-01-18,361.23,Quetzaltenango,Education
4,502916,Carlos Mendez,F,2026-01-29,339.01,Amatitlan,Retail
5,504200,Oscar Rojas,M,2026-01-09,138.22,Antigua,Retail
6,502029,Isabel Ramirez,M,2026-02-05,166.31,Villa Nueva,Services
7,501366,Pedro Herrera,M,2026-02-12,405.62,Amatitlan,Education
8,502614,Patricia Soto,F,2026-01-11,222.88,Guatemala,Retail
9,502794,Pablo Perez,M,2026-02-06,287.37,Escuintla,Retail
10,504410,Paola Herrera,F,2026-01-05,400.01,Antigua,Education


In [11]:
# Estructura y tipos de datos finales
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 4105 entries, 1 to 4999
Data columns (total 7 columns):
 #   Column          Non-Null Count  Dtype         
---  ------          --------------  -----         
 0   id_cliente      4105 non-null   int64         
 1   nombre          4105 non-null   object        
 2   genero          4105 non-null   object        
 3   fecha_registro  4105 non-null   datetime64[ns]
 4   gasto_q         4105 non-null   float64       
 5   ciudad          4105 non-null   object        
 6   categoria       4105 non-null   object        
dtypes: datetime64[ns](1), float64(1), int64(1), object(4)
memory usage: 256.6+ KB


In [12]:
# Resumen estadístico de las variables depuradas
df.describe()

,id_cliente,fecha_registro,gasto_q
count,4105.000000,4105,4105.000000
mean,502461.246772,2026-01-30 13:00:30.694275328,251.718195
min,500001.000000,2026-01-01 00:00:00,0.180000
25%,501252.000000,2026-01-16 00:00:00,126.410000
50%,502457.000000,2026-01-31 00:00:00,253.880000
75%,503687.000000,2026-02-14 00:00:00,378.440000
max,504900.000000,2026-03-01 00:00:00,499.860000
std,1413.956398,NaN,145.167263


## 7. Exploración mediante Tablas Pivote (Pandas)

Las tablas pivote permiten contrastar directamente la calidad y estructura de los datos antes y después del proceso de limpieza:

1. **Pivote A — Fragmentación Categórica (`ciudad` vs. `genero`):** Evidencia cómo 42 variantes con ruido de ciudades y 8 formatos de género se condensan en 8 ciudades canónicas y 2 géneros consistentes.
2. **Pivote B — Resumen de Calidad de Datos (Métricas ISO 25012):** Consolida los indicadores clave de calidad antes vs después.
3. **Pivote C — Valor Analítico Desbloqueado:** Presenta agregaciones cruzadas de gasto promedio y acumulado por ciudad y categoría, así como evolución mensual (análisis imposible de realizar en el estado original debido a tipos de datos texto e inconsistencias sintácticas).

In [13]:
# --- PIVOTE A: Fragmentación Categórica (Ciudad vs Género) ---
print(">> Tabla Pivote ANTES (Muestra del DataFrame Original con variantes y ruido):")
pivot_a_orig = pd.pivot_table(df_original, index='ciudad', columns='genero', values='id_cliente', aggfunc='count')
display(pivot_a_orig.head(15))
print(f"Variantes totales de ciudad en origen: {df_original['ciudad'].nunique(dropna=False)}")
print(f"Variantes totales de género en origen: {df_original['genero'].nunique(dropna=False)}")

>> Tabla Pivote ANTES (Muestra del DataFrame Original con variantes y ruido):


genero,,f,m,F,M,f,m
ciudad,,,,,,,
,1.0,16.0,10.0,14.0,12.0,6.0,9.0
Amatitlan,NaN,17.0,13.0,15.0,15.0,8.0,10.0
Antigua,NaN,9.0,7.0,12.0,8.0,9.0,9.0
Chimaltenango,1.0,10.0,6.0,6.0,12.0,11.0,6.0
Escuintla,1.0,8.0,9.0,10.0,7.0,6.0,8.0
Guatemala,NaN,8.0,13.0,9.0,10.0,6.0,7.0
Mixco,NaN,11.0,18.0,5.0,11.0,10.0,9.0
Quetzaltenango,2.0,12.0,8.0,16.0,10.0,11.0,12.0
Villa Nueva,1.0,7.0,17.0,7.0,13.0,10.0,9.0


Variantes totales de ciudad en origen: 42
Variantes totales de género en origen: 8


In [14]:
print(">> Tabla Pivote DESPUÉS (DataFrame Depurado y Canónico):")
pivot_a_clean = pd.pivot_table(df, index='ciudad', columns='genero', values='id_cliente', aggfunc='count', margins=True, margins_name='Total')
display(pivot_a_clean)
print(f"Ciudades canónicas: {df['ciudad'].nunique()} -> {sorted(df['ciudad'].unique())}")
print(f"Géneros canónicos: {df['genero'].nunique()} -> {sorted(df['genero'].unique())}")

>> Tabla Pivote DESPUÉS (DataFrame Depurado y Canónico):


genero,F,M,Total
ciudad,,,
Amatitlan,250,279,529
Antigua,267,253,520
Chimaltenango,237,247,484
Escuintla,257,227,484
Guatemala,262,260,522
Mixco,263,257,520
Quetzaltenango,273,262,535
Villa Nueva,257,254,511
Total,2066,2039,4105


Ciudades canónicas: 8 -> ['Amatitlan', 'Antigua', 'Chimaltenango', 'Escuintla', 'Guatemala', 'Mixco', 'Quetzaltenango', 'Villa Nueva']
Géneros canónicos: 2 -> ['F', 'M']


In [15]:
# --- PIVOTE B: Resumen de Calidad de Datos (ISO 25012) ---
calidad_df = pd.DataFrame({
    'Dimensión ISO 25012 / Métrica': [
        'Total de Registros (Completitud)',
        'Registros Duplicados (Unicidad)',
        'Nulos en Género (Completitud)',
        'Nulos en Ciudad (Completitud)',
        'Nulos en Gasto Q (Completitud)',
        'Variantes Únicas de Ciudad (Consistencia)',
        'Variantes Únicas de Género (Consistencia)',
        'Tipo de Dato Gasto (Exactitud sintáctica)',
        'Tipo de Dato Fecha (Exactitud sintáctica)',
        'Rango Temporal Válido (Consistencia temporal)'
    ],
    'Estado Original (Antes)': [
        f"{len(df_original):,}",
        f"{df_original.duplicated().sum()} ({df_original.duplicated().mean():.1%})",
        f"{nulos_reales['genero']} ({nulos_explicitos['genero']} NaN + {nulos_espacios['genero']} vacíos)",
        f"{nulos_reales['ciudad']} ({nulos_explicitos['ciudad']} NaN + {nulos_espacios['ciudad']} vacíos)",
        f"{nulos_reales['gasto_q']} ({nulos_reales['gasto_q'] / len(df_original.drop_duplicates()):.2%})",
        f"{df_original['ciudad'].nunique(dropna=False)} variantes con ruido",
        f"{df_original['genero'].nunique(dropna=False)} variantes (m, f, M, F, espacios)",
        'object (texto con comas decimales)',
        'object (formatos mixtos YYYY-MM-DD y DD/MM/YYYY)',
        'Inconsistente (fechas invertidas a dic/2026)'
    ],
    'Estado Depurado (Después)': [
        f"{len(df):,}",
        '0 (0.0%)',
        '0 (0.0%)',
        '0 (0.0%)',
        '0 (0.0%)',
        f"{df['ciudad'].nunique()} ciudades canónicas",
        f"{df['genero'].nunique()} géneros canónicos (F, M)",
        f"{df['gasto_q'].dtype} (punto decimal estándar)",
        f"{df['fecha_registro'].dtype} (estándar ISO)",
        f"{df['fecha_registro'].min().date()} a {df['fecha_registro'].max().date()} (100% verificado)"
    ]
})
display(calidad_df)

,Dimensión ISO 25012 / Métrica,Estado Original (Antes),Estado Depurado (Después)
0,Total de Registros (Completitud),"5,000","4,105"
1,Registros Duplicados (Unicidad),100 (2.0%),0 (0.0%)
2,Nulos en Género (Completitud),113 (60 NaN + 53 vacíos),0 (0.0%)
3,Nulos en Ciudad (Completitud),222 (152 NaN + 70 vacíos),0 (0.0%)
4,Nulos en Gasto Q (Completitud),497 (10.14%),0 (0.0%)
5,Variantes Únicas de Ciudad (Consistencia),42 variantes con ruido,8 ciudades canónicas
6,Variantes Únicas de Género (Consistencia),"8 variantes (m, f, M, F, espacios)","2 géneros canónicos (F, M)"
7,Tipo de Dato Gasto (Exactitud sintáctica),object (texto con comas decimales),float64 (punto decimal estándar)
8,Tipo de Dato Fecha (Exactitud sintáctica),object (formatos mixtos YYYY-MM-DD y DD/MM/YYYY),datetime64[ns] (estándar ISO)
9,Rango Temporal Válido (Consistencia temporal),Inconsistente (fechas invertidas a dic/2026),2026-01-01 a 2026-03-01 (100% verificado)


In [16]:
# --- PIVOTE C: Valor Analítico Desbloqueado ---
# Gasto promedio por Ciudad y Categoría
print(">> Gasto Promedio (Q) por Ciudad y Categoría:")
pivot_c_mean = pd.pivot_table(df, index='ciudad', columns='categoria', values='gasto_q', aggfunc='mean').round(2)
display(pivot_c_mean)

# Gasto total acumulado por Ciudad y Categoría
print("\n>> Gasto Total Acumulado (Q) por Ciudad y Categoría:")
pivot_c_sum = pd.pivot_table(df, index='ciudad', columns='categoria', values='gasto_q',
                             aggfunc='sum', margins=True, margins_name='Total').round(2)
display(pivot_c_sum)

# Agregación mensual de transacciones y montos
df['mes_registro'] = df['fecha_registro'].dt.to_period('M')
pivot_mensual = pd.pivot_table(df, index='mes_registro', columns='categoria', values='gasto_q', aggfunc=['count', 'sum']).round(2)
print("\n>> Transacciones y Gasto Acumulado por Mes y Categoría:")
display(pivot_mensual)

>> Gasto Promedio (Q) por Ciudad y Categoría:


categoria,Education,Food,Retail,Services
ciudad,,,,
Amatitlan,238.38,226.43,243.65,266.00
Antigua,244.15,252.60,258.14,225.31
Chimaltenango,264.75,263.15,271.65,261.35
Escuintla,269.44,261.80,269.02,278.58
Guatemala,248.02,250.65,225.96,237.08
Mixco,254.59,254.75,245.36,248.68
Quetzaltenango,238.17,280.21,258.33,256.53
Villa Nueva,245.84,243.35,254.18,227.21



>> Gasto Total Acumulado (Q) por Ciudad y Categoría:


categoria,Education,Food,Retail,Services,Total
ciudad,,,,,
Amatitlan,30512.22,31473.47,32161.29,34579.82,128726.80
Antigua,34913.91,34101.61,27362.57,30641.92,127020.01
Chimaltenango,31770.33,32367.68,34499.13,29794.32,128431.46
Escuintla,35296.25,30630.55,33358.41,31200.67,130485.88
Guatemala,29514.35,35090.99,29826.69,31057.28,125489.31
Mixco,30551.17,35919.09,29688.44,34318.26,130476.96
Quetzaltenango,29294.90,39789.84,36424.78,33092.78,138602.30
Villa Nueva,33434.35,28228.83,33552.05,28855.24,124070.47
Total,255287.48,267602.06,256873.36,253540.29,1033303.19



>> Transacciones y Gasto Acumulado por Mes y Categoría:


count                             sum                        \
categoria    Education Food Retail Services  Education       Food     Retail   
mes_registro                                                                   
2026-01            535  512    544      506  131458.41  133620.35  136848.12   
2026-02            472  518    456      497  120697.29  128206.50  116327.57   
2026-03             13   23     15       14    3131.78    5775.21    3697.67   

                         
categoria      Services  
mes_registro             
2026-01       123181.01  
2026-02       128103.50  
2026-03         2255.78

## 8. Interpretación de Resultados y Evaluación de Calidad

### Análisis del Impacto en Calidad de la Información (ISO/IEC 25012)
1. **Unicidad:**  
   Se eliminaron 100 registros idénticos (2.0% del dataset original), previniendo la duplicación artificial de transacciones y distorsión de métricas financieras.
2. **Completitud y Evaluación de la Estrategia de Nulos:**  
   Se identificaron 113 nulos reales en `genero`, 222 en `ciudad` y 497 en `gasto_q`. La aplicación de `dropna()` redujo el dataset de 4,900 a 4,105 registros (una reducción neta del 16.2%).  
   *Nota Metodológica:* Si bien la eliminación estricta garantiza 100% de completitud para análisis directos, en entornos de producción debe evaluarse la **imputación** (ej. mediana por categoría/ciudad para `gasto_q`, o asignar la categoría `'No Especificado'` en variables cualitativas) para conservar el 16.2% de volumen de registros si la pérdida de datos introduce sesgos de selección.
3. **Consistencia y Normalización Categórica:**  
   Se corrigió la dispersión de 42 formas erróneas de escritura de ciudades hacia 8 valores canónicos, y de 8 representaciones de género hacia 2 (`'F'` y `'M'`). Esto permitió que la tabla pivote de conteos refleje una distribución uniforme y fidedigna (529 en Amatitlán, 520 en Antigua, 484 en Chimaltenango, etc.).
4. **Exactitud Sintáctica y Temporalidad:**  
   La estandarización de `gasto_q` a tipo numérico `float64` y de `fecha_registro` a `datetime64[ns]` con `dayfirst=True` desbloqueó el cálculo de agregaciones financieras (`mean`, `sum`) y segmentaciones temporales mensuales, evitando el error de distorsión cronológica que situaba transacciones en diciembre de 2026.

## 9. Exportación de Datos Depurados

Exportamos el dataset depurado final en dos formatos estándar:
* **CSV (`limpio.csv`):** Formato universal en texto plano delimitado por comas.
* **Parquet (`limpio.parquet`):** Formato columnar binario de alto rendimiento, con compresión eficiente y preservación nativa de tipos de datos (`datetime64`, `float64`, `int64`).

In [17]:
# Preparación para exportación (excluyendo columnas auxiliares de análisis)
df_export = df.drop(columns=['mes_registro'], errors='ignore')

# 1. Exportación a CSV
df_export.to_csv("limpio.csv", index=False)
print("✓ Archivo 'limpio.csv' generado exitosamente (4,105 filas x 7 columnas).")

# 2. Exportación a Parquet
df_export.to_parquet("limpio.parquet", index=False, engine='pyarrow')
print("✓ Archivo 'limpio.parquet' generado exitosamente con PyArrow.")

✓ Archivo 'limpio.csv' generado exitosamente (4,105 filas x 7 columnas).
✓ Archivo 'limpio.parquet' generado exitosamente con PyArrow.
